In [1]:
import sys
from pathlib import Path

import torch

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

d = torch.load(PROJECT_ROOT / "data/vectors/qwen2.5-0.5b-instruct.pt")
v = d["V"][d["emotions"].index("desperate"), 16]   # hook model.model.layers[15]

In [2]:
from core.models import Model

m = Model()
m.load_weights()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [3]:
def add_steering(model, layer: int, v: torch.Tensor, alpha: float, scale: float):
    # v shape: d_model
    # add steering to the model
    # define a forward hook function that takes in X and adds a vector to it
        # scale the normalized vector with alpha and scale
    # index the layer from the model with layer to get the module (transformer block) to steer
    # register the forward hook on the module
    # return
    block = model.model.layers[layer - 1]
    direction = (v / v.norm()).to(model.device, model.dtype)
    hook_calls = {'steering_hook': 0}

    def steering_hook(module, input, output):
        h = output[0] if isinstance(output, tuple) else output
        h += alpha * scale * direction
        hook_calls['steering_hook'] += 1
        return output

    return block.register_forward_hook(steering_hook), hook_calls

In [4]:
from core.shards import read_shards

LAYER = 16

@torch.no_grad()
def find_steering_scale(model, tok, texts, layer, batch_size=16):
    tok.padding_side = "right"
    total, count = 0.0, 0
    for b in range(0, len(texts), batch_size):
        inputs = tok(texts[b:b + batch_size], padding=True, return_tensors="pt").to(model.device)
        h = model(**inputs, output_hidden_states=True).hidden_states[layer]
        mask = inputs.attention_mask.bool()
        total += h.norm(dim=-1)[mask].sum().item()
        count += mask.sum().item()
    return total / count

rows = read_shards(PROJECT_ROOT / "datasets/qwen-emotion-stories/corpus", sample=False)
neutrals = [r["text"] for r in rows if r["emotion"] == "neutral"]
scale = find_steering_scale(m.model, m.tok, neutrals, LAYER)   # ~49.0

In [16]:
# the paper's prompt, and next-token log-probs
text = m.tok.apply_chat_template(
    [{"role": "user", "content": "How does he feel?"}], tokenize=False, add_generation_prompt=True
) + "He feels"
inputs = m.tok(text, return_tensors="pt").to(m.device)

def gen_emotion(emotion = None, layer=16, alpha=0.5, max_tokens=20):
    if emotion is None:
        out = m.model.generate(**inputs, max_new_tokens=20, do_sample=False)
    else:
        i = d["emotions"].index(emotion)
        handle, _ = add_steering(m.model, layer, d["V"][i, layer], alpha, scale)
        try:
            out = m.model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
        finally:
            handle.remove()
    return m.tok.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)

In [19]:
d["emotions"]

['afraid',
 'angry',
 'ashamed',
 'calm',
 'desperate',
 'disgusted',
 'excited',
 'joyful',
 'lonely',
 'proud',
 'sad',
 'surprised']

In [37]:
gen_emotion("excited", alpha=0.4, max_tokens=30)

" excited! There's never a moment of excitement when you get the opportunity to become a superhero! Your heart jumps up in the blood! You can"